# 01 - Basics: Your First Iceberg Table

## What you'll learn
- How a namespace and table are created in a SQL catalog.
- How appending Arrow data creates Iceberg metadata and data files.
- How to read the same table through PyIceberg and DuckDB.

## Interview questions this answers
- What does an Iceberg table physically consist of?
- How do you read Iceberg without Spark?

In [ ]:
from datetime import datetime, timedelta

import duckdb
import pyarrow as pa
from pyiceberg.schema import Schema
from pyiceberg.types import DoubleType, LongType, NestedField, StringType, StructType, TimestampType

from src.catalog_helper import (
    configure_duckdb_for_minio,
    current_metadata_location,
    drop_table_if_exists,
    ensure_namespace,
    get_catalog,
)

catalog = get_catalog()

Create a clean table. Re-running this notebook drops only the demo table, not the whole catalog.

In [ ]:
ensure_namespace(catalog, "lab")
drop_table_if_exists(catalog, "lab.events")

schema = Schema(
    NestedField(field_id=1, name="event_id", field_type=StringType(), required=True),
    NestedField(field_id=2, name="event_ts", field_type=TimestampType(), required=True),
    NestedField(field_id=3, name="user_id", field_type=LongType(), required=False),
    NestedField(field_id=4, name="amount", field_type=DoubleType(), required=False),
    NestedField(
        field_id=5,
        name="location",
        field_type=StructType(
            NestedField(field_id=6, name="country", field_type=StringType()),
            NestedField(field_id=7, name="city", field_type=StringType()),
        ),
        required=False,
    ),
)

table = catalog.create_table("lab.events", schema=schema)
print("Table location:", table.location())
print("Metadata location:", current_metadata_location(table))

Notice the field IDs. They are the reason schema evolution can be safer than raw Parquet column-name matching.

In [ ]:
arrow_schema = pa.schema([
    pa.field("event_id", pa.string(), nullable=False),
    pa.field("event_ts", pa.timestamp("us"), nullable=False),
    pa.field("user_id", pa.int64()),
    pa.field("amount", pa.float64()),
    pa.field("location", pa.struct([
        pa.field("country", pa.string()),
        pa.field("city", pa.string()),
    ])),
])

start = datetime(2026, 5, 13, 10, 0, 0)
rows = [
    {
        "event_id": f"e{i:03d}",
        "event_ts": start + timedelta(minutes=i),
        "user_id": 100 + (i % 7),
        "amount": float((i % 5) * 10 + 0.99),
        "location": {"country": "DE" if i % 2 == 0 else "NL", "city": "Frankfurt" if i % 2 == 0 else "Amsterdam"},
    }
    for i in range(50)
]

table.append(pa.Table.from_pylist(rows, schema=arrow_schema))
table = catalog.load_table("lab.events")
print("Current metadata:", current_metadata_location(table))

Read through PyIceberg first. This path uses the catalog to find the current metadata file.

In [ ]:
arrow_result = table.scan().to_arrow()
arrow_result.to_pandas().head()

DuckDB reads through the Iceberg metadata JSON. On object storage, pointing directly at the current metadata file is more reliable than relying on `version-hint.text` discovery.

In [ ]:
con = duckdb.connect()
configure_duckdb_for_minio(con)
metadata_file = current_metadata_location(table)
con.execute(f"SELECT count(*) AS rows, min(amount) AS min_amount, max(amount) AS max_amount FROM iceberg_scan('{metadata_file}')").fetchdf()

## Try yourself
- In MinIO, open `warehouse/lab/events/` and count the files under `metadata/` and `data/`.
- Change the row count from 50 to 5, rerun, and explain which metadata files changed.